In [3]:
# Dépendances du notebook
%pip install openpyxl==3.1.3 pandas==3.0.3 s3fs==2026.3.0 -q


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


# Import des packages nécessaires

In [ ]:
import pandas as pd
import io
from openpyxl import load_workbook
from openpyxl.worksheet.formula import ArrayFormula
from openpyxl.styles import Font, PatternFill,Alignment,Border, Side
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.worksheet.worksheet import Worksheet
from openpyxl.chart import BarChart, PieChart, Reference
from openpyxl.chart.series import SeriesLabel
from openpyxl.utils import get_column_letter

# Import des données
Les données sont disponibles sur le site [DATA AMELI](https://data.ameli.fr/pages/home/).




J’ai choisi de travailler avec deux jeux de données totalement distincts :
**Une vue régionale et départementale**, décrivant les patients selon la pathologie, le traitement chronique ou l’épisode de soins, le sexe, la classe d’âge, la région et le département.  
  - Source : [Effectifs – Patients par pathologie](https://data.ameli.fr/explore/dataset/effectifs/information/)

**Une vue nationale**, portant sur les dépenses remboursées par pathologie et par patient en moyenne.  
  - Source : [Dépenses remboursées par pathologie](https://data.ameli.fr/explore/dataset/depenses/information/)

Comme le fichier **effectifs** est particulièrement volumineux, j’ai opté pour une **lecture en chunks**, ce qui permet de filtrer les données au fur et à mesure du chargement et d’éviter une surcharge mémoire.

J’ai ensuite enrichi ces données en les fusionnant avec un second fichier **region_dept** contenant les libellés des régions et des départements, afin d’obtenir un rendu plus harmonieux et plus lisible dans les visualisations.

Pour garantir un chargement fiable et éviter les problèmes liés aux chemins locaux, j’ai préféré déposer mes fichiers CSV sur Onyxia et les récupérer directement via leur URL MinIO. Cette approche assure un accès stable et reproductible aux données, quel que soit l’environnement d’exécution.

### Effectif de patients par pathologie, sexe, classe d'âge et territoire (département, région)

In [5]:
chunks = []

for chunk in pd.read_csv('https://minio.lab.sspcloud.fr/aissataa/PROJET_MEYTE/effectifs.csv', sep=';', chunksize=100_000,low_memory=False):
    filtered = chunk[(chunk['annee'] >= 2022) & (chunk['dept'] != '999')]
    chunks.append(filtered)

df_effectifs = pd.concat(chunks, ignore_index=True)

#Le datasaet qui va servir de jointure pour récuperer les libelles des départements et région
df_regions_dept = pd.read_csv(
    "https://minio.lab.sspcloud.fr/aissataa/PROJET_MEYTE/LibelleRegionDept.csv",
    sep=";",
    encoding="latin1",
    usecols=["departement", "libelle_departement", "libelle_region"]
)

# Harmonisation des colonnes , les 1 devient des 01 
df_regions_dept["departement"] = df_regions_dept["departement"].astype(str).str.zfill(2)
df_effectifs["dept"] = df_effectifs["dept"].astype(str).str.zfill(2)

# Jointure des 2 :
df_fusion = pd.merge(
    df_effectifs, 
    df_regions_dept, 
    left_on="dept", 
    right_on="departement", 
    how="inner"
)



Ici, je n’utilise pas `.copy()` car mon objectif n’est pas de dupliquer le DataFrame, mais simplement de le renommer.  
En faisant :

In [6]:
df_effectifs = df_fusion
del df_fusion # Et par la suite je supprime le dataFrame pour libérer de l'espace

## Nettoyage du dataset : df_effectifs

### Vérification et traitement des doublons

J’affiche ici les lignes du DataFrame qui apparaissent en doublon afin d’identifier
les répétitions éventuelles dans les données. L’option `keep=False` permet de visualiser
toutes les occurrences d’un doublon, ce qui facilite le diagnostic avant nettoyage.

Le fichier des correspondances *région–département* contient plusieurs doublons,ce qui génère des lignes dupliquées après la jointure.
Pour éviter ces répétitions dans le DataFrame final, j’ai décidé de supprimer les doublons avant de poursuivre l’analyse.


In [7]:
df_effectifs[df_effectifs.duplicated(keep=False)]


,annee,patho_niv1,patho_niv2,patho_niv3,top,cla_age_5,sexe,region,dept,Ntop,Npop,prev,Niveau prioritaire,libelle_classe_age,libelle_sexe,tri,libelle_region,departement,libelle_departement
0,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,RAR_MUC_IND,95et+,9,32,59,NaN,8460,NaN,3,plus de 95 ans,tous sexes,78.0,Hauts-de-France,59,Nord
1,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,RAR_MUC_IND,95et+,9,32,59,NaN,8460,NaN,3,plus de 95 ans,tous sexes,78.0,Hauts-de-France,59,Nord
2,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,RAR_MUC_IND,95et+,9,32,59,NaN,8460,NaN,3,plus de 95 ans,tous sexes,78.0,Hauts-de-France,59,Nord
3,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,RAR_MUC_IND,95et+,9,32,59,NaN,8460,NaN,3,plus de 95 ans,tous sexes,78.0,Hauts-de-France,59,Nord
4,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,RAR_MUC_IND,95et+,9,32,59,NaN,8460,NaN,3,plus de 95 ans,tous sexes,78.0,Hauts-de-France,59,Nord
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4878295,2023,Maladies cardioneurovasculaires,Artériopathie périphérique,Artériopathie périphérique,MCV_APE_IND,45-49,2,75,64,40.0,21590,0.185,"2,3",de 45 à 49 ans,femmes,27.0,Nouvelle-Aquitaine,64,Pyrénées-Atlantiques
4878296,2023,Maladies cardioneurovasculaires,Artériopathie périphérique,Artériopathie périphérique,MCV_APE_IND,45-49,2,75,64,40.0,21590,0.185,"2,3",de 45 à 49 ans,femmes,27.0,Nouvelle-Aquitaine,64,Pyrénées-Atlantiques
4878297,2023,Maladies cardioneurovasculaires,Artériopathie périphérique,Artériopathie périphérique,MCV_APE_IND,45-49,2,75,64,40.0,21590,0.185,"2,3",de 45 à 49 ans,femmes,27.0,Nouvelle-Aquitaine,64,Pyrénées-Atlantiques
4878298,2023,Maladies cardioneurovasculaires,Artériopathie périphérique,Artériopathie périphérique,MCV_APE_IND,45-49,2,75,64,40.0,21590,0.185,"2,3",de 45 à 49 ans,femmes,27.0,Nouvelle-Aquitaine,64,Pyrénées-Atlantiques


#### Nombre de doublons par colonne

In [8]:
df_effectifs.apply(lambda col: col.duplicated().sum())


annee                  4878298
patho_niv1             4878282
patho_niv2             4878251
patho_niv3             4878238
top                    4878221
cla_age_5              4878279
sexe                   4878297
region                 4878282
dept                   4878199
Ntop                   4869980
Npop                   4872749
prev                   4834295
Niveau prioritaire     4878294
libelle_classe_age     4878279
libelle_sexe           4878297
tri                    4878221
libelle_region         4878282
departement            4878199
libelle_departement    4878199
dtype: int64

### Poids du dataset avant et après suppression des doublons

Avant de nettoyer les données, j’affiche le poids mémoire du DataFrame afin d’évaluer l’impact des doublons sur la taille totale du fichier.  
Après suppression des lignes dupliquées, j’affiche à nouveau le poids du dataset pour mesurer le gain en mémoire et vérifier que le nettoyage a bien été appliqué.


In [9]:
buf = io.StringIO()
df_effectifs.info(buf=buf, verbose=False)
print("Poids AVANT nettoyage :", buf.getvalue().splitlines()[-1])



Poids AVANT nettoyage : memory usage: 707.2 MB


In [10]:
# Suppression des doublons
df_effectifs = df_effectifs.drop_duplicates()


In [11]:
# Afficher uniquement la ligne "memory usage"
buf = io.StringIO()
df_effectifs.info(buf=buf, verbose=False)
print("Poids APRÈS nettoyage :", buf.getvalue().splitlines()[-1])

Poids APRÈS nettoyage : memory usage: 141.4 MB


In [12]:
df_effectifs.head()


,annee,patho_niv1,patho_niv2,patho_niv3,top,cla_age_5,sexe,region,dept,Ntop,Npop,prev,Niveau prioritaire,libelle_classe_age,libelle_sexe,tri,libelle_region,departement,libelle_departement
0,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,RAR_MUC_IND,95et+,9,32,59,NaN,8460,NaN,3,plus de 95 ans,tous sexes,78.0,Hauts-de-France,59,Nord
5,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,RAR_MUC_IND,95et+,9,32,60,NaN,2470,NaN,3,plus de 95 ans,tous sexes,78.0,Hauts-de-France,60,Oise
10,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,RAR_MUC_IND,95et+,9,32,62,NaN,5010,NaN,3,plus de 95 ans,tous sexes,78.0,Hauts-de-France,62,Pas-de-Calais
15,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,RAR_MUC_IND,95et+,9,32,80,NaN,2220,NaN,3,plus de 95 ans,tous sexes,78.0,Hauts-de-France,80,Somme
20,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,RAR_MUC_IND,95et+,9,44,10,NaN,1570,NaN,3,plus de 95 ans,tous sexes,78.0,Grand Est,10,Aube


In [13]:
# Valeurs manquantes
(
    df_effectifs
    .isna()
    .sum(axis=0)
)

annee                       0
patho_niv1                  0
patho_niv2             101808
patho_niv3             220584
top                         0
cla_age_5                   0
sexe                        0
region                      0
dept                        0
Ntop                   259584
Npop                        0
prev                   259584
Niveau prioritaire      12726
libelle_classe_age          0
libelle_sexe                0
tri                     12726
libelle_region              0
departement                 0
libelle_departement         0
dtype: int64

Dans ce jeu de données, les valeurs indiquées comme NaN ne correspondent pas à de véritables valeurs manquantes. Elles apparaissent simplement lorsqu’un département ne présente aucun cas pour une pathologie donnée. Dans ce contexte, il est donc logique de remplacer ces NaN par 0, afin de refléter correctement l’absence de cas observés.

In [14]:
df_effectifs = df_effectifs.fillna(0)


###  Après avoir passer les NAN à 0

In [15]:
# Valeurs manquantes
(
    df_effectifs
    .isna()
    .sum(axis=0)
)

annee                  0
patho_niv1             0
patho_niv2             0
patho_niv3             0
top                    0
cla_age_5              0
sexe                   0
region                 0
dept                   0
Ntop                   0
Npop                   0
prev                   0
Niveau prioritaire     0
libelle_classe_age     0
libelle_sexe           0
tri                    0
libelle_region         0
departement            0
libelle_departement    0
dtype: int64

### Nettoyage et préparation des données

- J’ai décidé de supprimer les parenthèses dans les variables *patho_niv2* et *patho_niv3*, car elles entraînaient un affichage peu lisible dans les visualisations et ajoutaient trop d’informations inutiles.
- Je me suis concentrée sur la France hexagonale, en incluant la Corse et les DOM.
- Les codes **FRANCE** et **Tout département** correspondent à des agrégats régionaux ou nationaux (valeurs non rattachées à un département).  
  Comme ils faussent l’analyse territoriale, j’ai choisi de les supprimer.


In [16]:
# Nettoyage des parenthèses
cols = ["patho_niv2", "patho_niv3"]
for c in cols:
    df_effectifs.loc[:, c] = df_effectifs[c].str.replace(r"\s*\(.*\)", "", regex=True)

# Grand filtre de nettoyage , je veux gardé que la france hexagonale
hors_hexa = ['Tout département', 'Guadeloupe ', 'Haute-Corse', 'Martinique', 'La Réunion', 'Guyane', 'Mayotte', 'Corse-du-Sud','FRANCE']

df_effectifs = df_effectifs[
    (~df_effectifs['libelle_departement'].astype(str).isin(hors_hexa)) &
    
    # On enlève la région "France" qui englobe tout
    (df_effectifs['libelle_region'].astype(str) != 'FRANCE')
]


Pour ne conserver que les observations réellement exploitables, je filtre le dataset
en gardant uniquement les lignes où la variable `prev` est strictement supérieure à 0.
Cela permet d’éliminer les enregistrements sans prévalence et d’alléger les analyses
et visualisations suivantes.

In [17]:
df_effectifs = df_effectifs[df_effectifs["prev"] > 0]


In [18]:
df_effectifs.dtypes


annee                    int64
patho_niv1                 str
patho_niv2              object
patho_niv3              object
top                        str
cla_age_5                  str
sexe                     int64
region                   int64
dept                       str
Ntop                   float64
Npop                     int64
prev                   float64
Niveau prioritaire      object
libelle_classe_age         str
libelle_sexe               str
tri                    float64
libelle_region             str
departement                str
libelle_departement        str
dtype: object

### Nettoyage des colonnes `patho_niv2` et `patho_niv3` après remplacement des `NaN`

Lors du remplacement des valeurs manquantes (`NaN`) par `0`, les colonnes `patho_niv2` et
`patho_niv3` ont changé de type : elles sont passées du type *string* au type *object*.
Lorsqu’une colonne contient un mélange de valeurs c'est ce que pandas fait numériques, textuelles ou des `NaN`, elle est automatiquement convertie en `object`.
Comme les `NaN` avaient été remplacés par `0`, ces valeurs sont restées sous forme de `"0"` après conversion en `str`.  
Je les ai supprimé pour ne conserver que les niveaux de pathologie valides.


In [19]:
df_effectifs["patho_niv2"] = df_effectifs["patho_niv2"].astype(str).str.strip()
df_effectifs["patho_niv3"] = df_effectifs["patho_niv3"].astype(str).str.strip()


In [20]:
df_effectifs.dtypes


annee                    int64
patho_niv1                 str
patho_niv2                 str
patho_niv3                 str
top                        str
cla_age_5                  str
sexe                     int64
region                   int64
dept                       str
Ntop                   float64
Npop                     int64
prev                   float64
Niveau prioritaire      object
libelle_classe_age         str
libelle_sexe               str
tri                    float64
libelle_region             str
departement                str
libelle_departement        str
dtype: object

In [21]:
df_effectifs = df_effectifs[
    (df_effectifs["patho_niv1"] != "0") &
    (df_effectifs["patho_niv2"] != "0") &
    (df_effectifs["patho_niv3"] != "0")
]

### Suppression des colonnes non utilisées

Dans cette étape, je retire les colonnes qui ne sont pas nécessaires pour l’analyse finale.  
Il s’agit principalement de variables techniques, de codes intermédiaires ou de niveaux d’agrégation
qui ne sont pas exploités dans les visualisations (ex. : codes de tri, niveaux prioritaires, variables
de regroupement, ou encore des informations redondantes comme la colonne **region** issue d’un des
datasets utilisés pour la jointure).

L’option `errors='ignore'` permet d’éviter une erreur si certaines colonnes ont déjà été supprimées
lors d’étapes précédentes du nettoyage.


In [22]:
# Suppression des colonnes inutiles pour le nettoyage final
df_effectifs = df_effectifs.drop(
    columns=[
        "Code département",
        "tri",
        "Niveau prioritaire",
        "region",
        "dept",
        "departement",
        "cla_age_5",
        "top",
        "sexe",

    ],
    errors='ignore' # Évite de faire une erreur si ça a déjà été supprimée
)




### Renommage de certaines colonnes

Pour améliorer la lisibilité du dataset et faciliter la compréhension des analyses,
j’ai choisi de renommer plusieurs colonnes.  
Afin d’obtenir des intitulés plus courts, plus explicites et cohérents avec les visualisations produites par la suite.



In [23]:
# Renommage des colonens
df_effectifs = df_effectifs.rename(columns={
    "libelle_departement": "Département",
    "libelle_region": "Région",
    "prev" : "Prévalence",
    "Npop": "Population de référence",
    "Ntop": "Effectif",
    "libelle_sexe" : "Sexe",
    "libelle_classe_age" : "Classe d'age"


})

In [24]:
df_effectifs = df_effectifs.dropna()

In [25]:
df_effectifs

,annee,patho_niv1,patho_niv2,patho_niv3,Effectif,Population de référence,Prévalence,Classe d'age,Sexe,Région,Département
150,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,150.0,981490,0.015,tous âges,hommes,Ile-de-France,Paris
155,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,30.0,204050,0.015,tous âges,hommes,Centre-Val de Loire,Eure-et-Loir
160,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,20.0,123680,0.016,tous âges,hommes,Bourgogne-Franche-Comté,Jura
170,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,40.0,259740,0.014,tous âges,hommes,Bourgogne-Franche-Comté,Saône-et-Loire
175,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,10.0,65260,0.018,tous âges,hommes,Bourgogne-Franche-Comté,Territoire de Belfort
...,...,...,...,...,...,...,...,...,...,...,...
4878265,2023,Maladies cardioneurovasculaires,Artériopathie périphérique,Artériopathie périphérique,40.0,47150,0.091,de 45 à 49 ans,femmes,Pays de la Loire,Loire-Atlantique
4878270,2023,Maladies cardioneurovasculaires,Artériopathie périphérique,Artériopathie périphérique,40.0,21550,0.162,de 45 à 49 ans,femmes,Pays de la Loire,Vendée
4878275,2023,Maladies cardioneurovasculaires,Artériopathie périphérique,Artériopathie périphérique,30.0,17860,0.174,de 45 à 49 ans,femmes,Bretagne,Côtes-d'Armor
4878280,2023,Maladies cardioneurovasculaires,Artériopathie périphérique,Artériopathie périphérique,40.0,34710,0.107,de 45 à 49 ans,femmes,Bretagne,Ille-et-Vilaine


### Exploration des niveaux de pathologies

Avant d’appliquer des filtres plus précis, j’affiche les valeurs uniques de `patho_niv2` et `patho_niv3`
(en excluant les valeurs `NaN`). Cela permet de vérifier la cohérence des libellés et d’identifier
éventuels regroupements ou corrections à effectuer avant l’analyse.


In [26]:
print(df_effectifs['patho_niv1'].dropna().unique())
print(df_effectifs['patho_niv2'].dropna().unique())
print(df_effectifs['patho_niv3'].dropna().unique())


<StringArray>
[                                                               'Maladies inflammatoires ou rares ou infection VIH',
                                                                                           'Maladies neurologiques',
                                                                                          'Maladies psychiatriques',
 'Pas de pathologie repérée, traitement, maternité, hospitalisation ou traitement antalgique ou anti-inflammatoire',
                                                                                   'Total consommants tous régimes',
                                                              'Traitements du risque vasculaire (hors pathologies)',
                                                                      'Traitements psychotropes (hors pathologies)',
                                                                                  'Maladies cardioneurovasculaires',
                                                  

# 2 ème dataset que je vais étudier :

# Dépenses remboursées affectées à chaque pathologie

Les données présentent des informations sur les dépenses remboursées par l’ensemble des régimes d’assurance maladie et affectées à chaque pathologie, traitement chronique ou épisode de soins. Ces dépenses sont réparties en trois grandes catégories :

* les soins de ville (consultations médicales, soins infirmiers, actes de kinésithérapie, médicaments, biologie, transports, etc.)  ; 

* les hospitalisations dans les établissements de santé publics ou privés ;

* les prestations en espèces, notamment les indemnités journalières.

## Deux types d’indicateurs sont proposés pour chacune de ces catégories : 

* les dépenses totales annuelles remboursées, qui mesurent le poids financier global d’une pathologie ;

* les dépenses moyennes annuelles remboursées par patient, qui permettent d’apprécier le coût moyen associé à chaque pathologie.

Ces deux indicateurs offrent ainsi une vision complémentaire des dépenses de santé, en combinant à la fois l’impact global et la charge moyenne par patient.

In [27]:
chunks = []

for chunk in pd.read_csv(
    'https://minio.lab.sspcloud.fr/aissataa/PROJET_MEYTE/depenses.csv',
    sep=';',
    chunksize=100_000,
    low_memory=False
):
    filtered = chunk[
        (chunk['annee'] >= 2022) &
        (chunk['montant'] > 0) 
    ]
    chunks.append(filtered)

df_depenses = pd.concat(chunks, ignore_index=True)
df_depenses


,annee,patho_niv1,patho_niv2,patho_niv3,top,dep_niv_1,dep_niv_2,montant,Ntop,N_recourant_au_poste,montant_moy,Niveau prioritaire,tri,type_somme
0,2023,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,ALD_CAT_CAT,Hospitalisations (tous secteurs),Hospitalisations liste en sus MCO secteur priv...,3856836,1714310.0,34840.0,2.0,"1,2,3",16.0,Partiel
1,2023,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,ALD_CAT_CAT,Hospitalisations (tous secteurs),Total hospitalisations (tous secteurs) rembour...,308350458,1714310.0,1232630.0,180.0,"1,2,3",16.0,Total
2,2023,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,ALD_CAT_CAT,Prestations en espèces,Indemnités journalières maladie et AT/MP rembo...,117367790,1714310.0,125760.0,68.0,"1,2,3",16.0,Partiel
3,2023,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,ALD_CAT_CAT,Prestations en espèces,Prestations d'invalidité remboursées,263517389,1714310.0,68580.0,154.0,"1,2,3",16.0,Partiel
4,2023,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,ALD_CAT_CAT,Soins de ville,Autres dépenses de soins de ville remboursés,9004352,1714310.0,424990.0,5.0,"1,2,3",16.0,Partiel
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4265,2022,Traitements psychotropes (hors pathologies),Traitements neuroleptiques (hors pathologies),Traitements neuroleptiques (hors pathologies),TPS_NLE_EXC,Soins de ville,Autres produits de santé remboursés,10878069,335640.0,212200.0,32.0,"2,3",55.0,Partiel
4266,2022,Traitements psychotropes (hors pathologies),Traitements neuroleptiques (hors pathologies),Traitements neuroleptiques (hors pathologies),TPS_NLE_EXC,Soins de ville,Soins de généralistes remboursés,11874138,335640.0,300910.0,35.0,"2,3",55.0,Partiel
4267,2022,Traitements psychotropes (hors pathologies),Traitements neuroleptiques (hors pathologies),Traitements neuroleptiques (hors pathologies),TPS_NLE_EXC,Soins de ville,Soins dentaires remboursés,6030821,335640.0,118450.0,18.0,"2,3",55.0,Partiel
4268,2022,Traitements psychotropes (hors pathologies),Traitements neuroleptiques (hors pathologies),Traitements neuroleptiques (hors pathologies),TPS_NLE_EXC,Soins de ville,Total soins de ville remboursés,199465569,335640.0,335630.0,594.0,"2,3",55.0,Total


# Les champs sont les suivants avant nettoyage:

In [28]:
df_depenses.columns


Index(['annee', 'patho_niv1', 'patho_niv2', 'patho_niv3', 'top', 'dep_niv_1',
       'dep_niv_2', 'montant', 'Ntop', 'N_recourant_au_poste', 'montant_moy',
       'Niveau prioritaire', 'tri', 'type_somme'],
      dtype='str')

In [29]:
# Suppression des colonnes inutiles pour le nettoyage final
df_depenses = df_depenses.drop(
    columns=[
        "tri",
        "Niveau prioritaire",
        "type_somme",
        "top",

    ],
    errors='ignore' # Évite de faire une erreur si ça a déjà été supprimée
)




# Nettoyage des données

In [30]:
(
    df_depenses
    .isna()
    .sum(axis=0)
)

annee                     0
patho_niv1                0
patho_niv2              457
patho_niv3              997
dep_niv_1                 0
dep_niv_2                 0
montant                   0
Ntop                     28
N_recourant_au_poste     28
montant_moy              28
dtype: int64

In [31]:
df_depenses = df_depenses.drop_duplicates()


In [32]:
df_depenses


,annee,patho_niv1,patho_niv2,patho_niv3,dep_niv_1,dep_niv_2,montant,Ntop,N_recourant_au_poste,montant_moy
0,2023,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Hospitalisations (tous secteurs),Hospitalisations liste en sus MCO secteur priv...,3856836,1714310.0,34840.0,2.0
1,2023,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Hospitalisations (tous secteurs),Total hospitalisations (tous secteurs) rembour...,308350458,1714310.0,1232630.0,180.0
2,2023,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Prestations en espèces,Indemnités journalières maladie et AT/MP rembo...,117367790,1714310.0,125760.0,68.0
3,2023,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Prestations en espèces,Prestations d'invalidité remboursées,263517389,1714310.0,68580.0,154.0
4,2023,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Soins de ville,Autres dépenses de soins de ville remboursés,9004352,1714310.0,424990.0,5.0
...,...,...,...,...,...,...,...,...,...,...
4265,2022,Traitements psychotropes (hors pathologies),Traitements neuroleptiques (hors pathologies),Traitements neuroleptiques (hors pathologies),Soins de ville,Autres produits de santé remboursés,10878069,335640.0,212200.0,32.0
4266,2022,Traitements psychotropes (hors pathologies),Traitements neuroleptiques (hors pathologies),Traitements neuroleptiques (hors pathologies),Soins de ville,Soins de généralistes remboursés,11874138,335640.0,300910.0,35.0
4267,2022,Traitements psychotropes (hors pathologies),Traitements neuroleptiques (hors pathologies),Traitements neuroleptiques (hors pathologies),Soins de ville,Soins dentaires remboursés,6030821,335640.0,118450.0,18.0
4268,2022,Traitements psychotropes (hors pathologies),Traitements neuroleptiques (hors pathologies),Traitements neuroleptiques (hors pathologies),Soins de ville,Total soins de ville remboursés,199465569,335640.0,335630.0,594.0


# Filtrage avant creation du dashboard

In [33]:
df_depenses = df_depenses.drop('Total consommants tous régimes', errors='ignore')

# Création du fichier Excel

Je vais créer un nouveau fichier Excel qui contiendra les données nettoyées dans un onglet "cleanedData_depenses" et "cleanedData_Pathos"

Pour les dépenses et pour les pathologies :

In [34]:
output_dep = '../DATA/Dashboard_Depenses.xlsx'

with pd.ExcelWriter(output_dep, engine='openpyxl') as writer:
    df_depenses.to_excel(writer, sheet_name='cleanedData_depenses', index=False)

print(f"Fichier dépenses créé : {output_dep}")


Fichier dépenses créé : ../DATA/Dashboard_Depenses.xlsx


In [35]:
output_eff = '../DATA/Dashboard_Effectifs.xlsx'

with pd.ExcelWriter(output_eff, engine='openpyxl') as writer:
    df_effectifs.to_excel(writer, sheet_name='cleanedData_Effectifs', index=False)

print(f"Fichier effectifs créé : {output_eff}")


Fichier effectifs créé : ../DATA/Dashboard_Effectifs.xlsx


In [36]:
wb_eff = load_workbook(output_eff)

In [37]:
wb_dep = load_workbook(output_dep) 

Calcul des listes _ len_dict

In [ ]:
annees = sorted(df_depenses['annee'].dropna().unique())
pathos = sorted(df_depenses['patho_niv1'].dropna().unique())
postes = sorted(df_depenses['dep_niv_1'].dropna().unique())

# LISTES POUR FILTRES EFFECTIFS
depts = sorted(df_effectifs['Département'].dropna().unique())
classes_age = sorted(df_effectifs['Classe d\'age'].dropna().unique())
sexes = sorted(df_effectifs['Sexe'].dropna().unique())
regions = sorted(df_effectifs['Région'].dropna().unique())
pathos_eff = sorted(df_effectifs['patho_niv1'].dropna().unique())

# LEN_DICT
len_dict = {
    'len_annee': len(annees) + 1,
    'len_patho_niv1': len(pathos) + 1,
    'len_dep_niv_1': len(postes) + 1,
    'len_departement_eff': len(depts) + 1,
    'len_classes_age': len(classes_age) + 1,
    'len_sexe': len(sexes) + 1,
    'len_region': len(regions) + 1,
    'len_patho_effectif': len(pathos_eff) + 1, 
}

import pprint
pprint.pprint(len_dict)

{'len_annee': 3,
 'len_classes_age': 22,
 'len_dep_niv_1': 5,
 'len_departement_eff': 96,
 'len_patho_effectif': 19,
 'len_patho_niv1': 20,
 'len_region': 14,
 'len_sexe': 4}


In [ ]:
cols_to_calculate = ['annee', 'patho_niv1', 'dep_niv_1', 'dep_niv_2']

cols_to_calculate_effectifs = {
    'annee_eff': 'annee',
    'patho_effectif': 'patho_niv1',
    'age': "Classe d'age",
    'sexe': 'Sexe',
    'region': 'Région',
    'departement_eff': 'Département'
}
len_dict = {}

# BOUCLE POUR DÉPENSES
for col in cols_to_calculate:
    if col == 'patho_niv1':
        # Exclure "Total"
        uniques = df_depenses.loc[
            ~df_depenses[col].isin(['Total', 'Total consommants tous régimes']),
            col
        ].dropna().unique()
    else:
        uniques = df_depenses[col].dropna().unique()
    
    len_dict[f"len_{col}"] = len(uniques) + 1

# BOUCLE POUR EFFECTIFS
for key_name, col_real_name in cols_to_calculate_effectifs.items():
    if col_real_name in df_effectifs.columns:
        if key_name == 'age':
            # Exclure 'tous âges', 'tous ages', 'total'
            uniques = df_effectifs.loc[
                ~df_effectifs[col_real_name].astype(str).str.lower().isin(['tous âges', 'tous ages', 'total']),
                col_real_name
            ].dropna().unique()
        elif key_name == 'patho_effectif':
            # Exclure 'total'
            uniques = df_effectifs.loc[
                ~df_effectifs[col_real_name].astype(str).str.lower().isin(['total']),
                col_real_name
            ].dropna().unique()
        else:
            uniques = df_effectifs[col_real_name].dropna().unique()
        
        len_dict[f"len_{key_name}"] = len(uniques) + 1

# Dépenses
annees = sorted(df_depenses['annee'].dropna().unique())

pathos = sorted(df_depenses.loc[
    ~df_depenses['patho_niv1'].isin(['Total', 'Total consommants tous régimes']),
    'patho_niv1'
].dropna().unique())

postes = sorted(df_depenses['dep_niv_1'].dropna().unique())
postes_niv2 = sorted(df_depenses['dep_niv_2'].dropna().unique())

# Effectifs
depts = sorted(df_effectifs['Département'].dropna().unique())

classes_age = sorted(df_effectifs.loc[
    ~df_effectifs['Classe d\'age'].astype(str).str.lower().isin(['tous âges', 'tous ages', 'total']),
    'Classe d\'age'
].dropna().unique())

sexes = sorted(df_effectifs['Sexe'].dropna().unique())
regions = sorted(df_effectifs['Région'].dropna().unique())

pathos_eff = sorted(df_effectifs.loc[
    ~df_effectifs['patho_niv1'].astype(str).str.lower().isin(['Total', 'Total consommants tous régimes']),
    'patho_niv1'
].dropna().unique())

pprint.pprint(len_dict)

{'len_age': 21,
 'len_annee': 3,
 'len_annee_eff': 3,
 'len_dep_niv_1': 5,
 'len_dep_niv_2': 32,
 'len_departement_eff': 96,
 'len_patho_effectif': 19,
 'len_patho_niv1': 19,
 'len_region': 14,
 'len_sexe': 4}


Création d'un onglet Filtres

Cet onglet est nécessaire pour alimenter les listes de validaiton dans les filtres.

Je vais récupérer l'année, la pathologie et le poste de dépense avec la fonction UNIQUE() depuis les onglets "cleaned_data"

In [40]:
from openpyxl.utils import FORMULAE
"UNIQUE" in FORMULAE

False

In [51]:
# DÉFINITION DES NOMS DE ONGLETS SOURCES
sheet_dep = 'cleanedData_depenses'
sheet_eff = 'cleanedData_Effectifs'

#CHARGEMENT DES WORKBOOKS
wb_eff = load_workbook(output_eff)
wb_dep = load_workbook(output_dep)

# GESTION DE L'ONGLET FILTRES
if 'Filtres' not in wb_eff.sheetnames:
    filtres_sheet = wb_eff.create_sheet('Filtres', 0)
else:
    filtres_sheet = wb_eff['Filtres']
    filtres_sheet.delete_rows(1, filtres_sheet.max_row)

# ARRAYFORMULAS EXCEL

# Formules basées sur les dépenses
formula_annee = f"=_xlfn.SORT(_xlfn.UNIQUE({sheet_dep}!A:A))"
filtres_sheet['A1'] = ArrayFormula(f"A1:A{len_dict['len_annee']}", formula_annee)

formula_patho = f"=_xlfn.UNIQUE({sheet_dep}!B:B)"
filtres_sheet['B1'] = ArrayFormula(f"B1:B{len_dict['len_patho_niv1']}", formula_patho)

formula_poste = f"=_xlfn.UNIQUE({sheet_dep}!E:E)"
filtres_sheet['C1'] = ArrayFormula(f"C1:C{len_dict['len_dep_niv_1']}", formula_poste)

formula_poste_niv2 = f"=_xlfn.UNIQUE({sheet_dep}!F:F)"
filtres_sheet['D1'] = ArrayFormula(f"D1:D{len_dict.get('len_dep_niv_2', len_dict['len_dep_niv_1'])}", formula_poste_niv2)

# Formules basées sur les effectifs
formula_annee_eff = f"=_xlfn.UNIQUE({sheet_eff}!A:A)"
filtres_sheet['E1'] = ArrayFormula(f"E1:E{len_dict.get('len_annee_eff', len_dict['len_annee'])}", formula_annee_eff)

formula_patho_eff = f"=_xlfn.UNIQUE({sheet_eff}!B:B)"
filtres_sheet['F1'] = ArrayFormula(f"F1:F{len_dict.get('len_patho_effectif', len_dict['len_patho_effectif'])}", formula_patho_eff)

formula_age = f"=_xlfn.UNIQUE({sheet_eff}!H:H)"
filtres_sheet['G1'] = ArrayFormula(f"G1:G{len_dict.get('len_age', len_dict['len_classes_age'])}", formula_age)

formula_sexe = f"=_xlfn.UNIQUE({sheet_eff}!I:I)"
filtres_sheet['H1'] = ArrayFormula(f"H1:H{len_dict['len_sexe']}", formula_sexe)

formula_region = f"=_xlfn.UNIQUE({sheet_eff}!J:J)"
filtres_sheet['I1'] = ArrayFormula(f"I1:I{len_dict['len_region']}", formula_region)

formula_dept = f"=_xlfn.UNIQUE({sheet_eff}!K:K)"
filtres_sheet['J1'] = ArrayFormula(f"J1:J{len_dict['len_departement_eff']}", formula_dept)

## Fonctions utilitaires

Comme nous allons répéter cette opération plusieurs fois, il est préférable de créer une fonction qui effectue les mêmes opérations.

In [48]:
def auto_adjust_columns(sheet: Worksheet) -> None:
    """Ajuste automatiquement la largeur des colonnes selon le contenu."""
    for col_idx, column in enumerate(sheet.columns, 1):
        max_length = 0
        col_letter = get_column_letter(col_idx)
        for cell in column:
            if cell.value:
                max_length = max(max_length, len(str(cell.value)))
        # Ajustement propre : max_length + 2 pour respirer, plafonné à 50
        adjusted_width = min(max_length + 2, 50)
        sheet.column_dimensions[col_letter].width = max(adjusted_width, 12)


def add_filter_title(sheet: Worksheet, title: str, start_row: int, start_column: int) -> None:
    """Ajoute le titre du filtre avec la mise en forme."""
    cell = sheet.cell(row=start_row, column=start_column)
    cell.value = title
    cell.alignment = Alignment(horizontal='center', vertical='center')
    cell.fill = PatternFill(start_color='FF1F4E78', fill_type='solid') 
    cell.font = Font(bold=True)


def add_filter_value(sheet: Worksheet, value, start_row: int, start_column: int) -> None:
    """Ajoute la valeur par défaut du filtre."""
    cell = sheet.cell(row=start_row, column=start_column)
    cell.value = value
    cell.alignment = Alignment(horizontal='center', vertical='center')
    cell.fill = PatternFill(start_color='FF1F4E78', fill_type='solid')
    cell.font = Font(bold=True)


def add_data_validation(sheet: Worksheet, start_row: int, start_column: int, formula: str) -> None:
    """Ajoute la liste déroulante (Data Validation) sur la cellule cible."""
    dv = DataValidation(type='list', formula1=formula)
    sheet.add_data_validation(dv)
    dv.add(sheet.cell(row=start_row, column=start_column).coordinate)


def add_filter(sheet: Worksheet, title: str, value, start_row: int, start_column: int, end_row: int, end_column: int, formula: str) -> None:
    """Construit un bloc filtre complet avec cellules fusionnées et liste déroulante."""
    target_val_column = end_column + 1
    
    add_filter_title(sheet, title, start_row, start_column)
    add_filter_value(sheet, value, start_row, target_val_column)
    add_data_validation(sheet, start_row, target_val_column, formula)
    
    sheet.merge_cells(start_row=start_row, start_column=start_column, end_row=end_row, end_column=end_column)
    sheet.merge_cells(start_row=start_row, start_column=target_val_column, end_row=end_row, end_column=target_val_column + 1)

# Charger les deux fichiers séparés
wb_dep = load_workbook(output_dep)
wb_eff = load_workbook(output_eff)

# Sauvegarde indépendante des deux classeurs (Juste avant l'étape de fusion finale)
wb_dep.save(output_dep)
wb_eff.save(output_eff)

wb_dep.close()
wb_eff.close()

Création du dashboard

Je vais créer un premier tableau de bord interactif avec des filtres.
Je vais me concentrer sur les dépenses de remboursement par pathologie et par poste de dépense.
L'utilisateur aura la possibilité de choisir l'année, la pathologie et le poste de dépense via des filtres sous forme de listes de validation.
Ces filtres permettront d'actualiser dynamiquement les formules et les graphiques.

Design :

In [54]:
wb_eff = load_workbook(output_eff)
wb_dep = load_workbook(output_dep)
# === COULEURS ===
COULEURS = {
    'bleu_fonce': 'FF1F4E78',
    'bleu_clair': 'FF4472C4',
    'cyan': 'FF00CCFF',
    'jaune_filtre': 'FFF2CC',
    'gris_clair': 'FFCCCCCC',
}

# Noms des sheets d'origine
sheet_dep = 'cleanedData_depenses'
sheet_eff = 'cleanedData_Effectifs'



## Onglet postes

In [55]:
wb_dep = load_workbook(output_dep)

# ONGLET POSTES
# === 1. ACCÈS OU CRÉATION DE L'ONGLET 'POSTES' ===
if 'Postes' not in wb_dep.sheetnames:
    postes_sheet = wb_dep.create_sheet('Postes', 1)
else:
    postes_sheet = wb_dep['Postes']
    postes_sheet.delete_rows(1, postes_sheet.max_row)

postes_sheet.sheet_view.showGridLines = False

# === 2. FILTRE DE SÉLECTION D'ANNÉE ===
postes_sheet['A1'] = 'Année'
postes_sheet['A1'].font = Font(bold=True, color='FFFFFF')
postes_sheet['A1'].fill = PatternFill(start_color=COULEURS['bleu_fonce'], fill_type='solid')

postes_sheet['C1'] = int(annees[-1])
postes_sheet['C1'].font = Font(bold=True)
postes_sheet['C1'].fill = PatternFill(start_color=COULEURS['jaune_filtre'], fill_type='solid')

dv_annee_postes = DataValidation(type='list', formula1=f"=Filtres!$A$2:$A${len_dict['len_annee']}")
wb_dep['Postes'].add_data_validation(dv_annee_postes)
dv_annee_postes.add('C1')

# Bannière décorative sous le filtre
postes_sheet['A3'] = ""
postes_sheet.merge_cells('A3:E3')
postes_sheet['A3'].fill = PatternFill(start_color=COULEURS['bleu_fonce'], fill_type='solid')

# Card du KPI Total Dépenses globales
postes_sheet['A5'] = "Total dépenses"
postes_sheet['A5'].font = Font(bold=True, size=11)
postes_sheet['A6'] = f"=SUM(B9:B30)"  # Ajusté dynamiquement selon tes formules
postes_sheet['A6'].font = Font(bold=True, size=14, color=COULEURS['bleu_fonce'])
postes_sheet['A6'].number_format = '#,##0'

# === 3. EN-TÊTES DE LA TABLE DE DIAGNOSTIC (Ligne 8) ===
postes_sheet['A8'] = "Poste (dep_niv_1)"  # En-tête explicite demandé
postes_sheet['B8'] = "Dépenses"
postes_sheet['C8'] = "Patients"
postes_sheet['D8'] = "Coût/patient"

for col in ['A', 'B', 'C', 'D']:
    postes_sheet[f'{col}8'].font = Font(bold=True, color='FFFFFF')
    postes_sheet[f'{col}8'].fill = PatternFill(start_color=COULEURS['bleu_clair'], fill_type='solid')

# === 4. INJECTION DES LIGNES ET FORMULES DE CALCUL (Ligne 9+) ===
row_poste = 9

# Filtrage pour enlever d'éventuels totaux génériques présents dans le mapping
postes_valides = [p for p in postes if p and "total" not in p.lower()]

for poste in postes_valides:
    # Colonne A : Nom du grand poste (dep_niv_1)
    postes_sheet[f'A{row_poste}'] = poste
    
    # Colonne B : Somme des Dépenses (Somme de G conditionnée par l'année C1 et le poste A9+)
    postes_sheet[f'B{row_poste}'] = f"=IFERROR(SUMIFS('{sheet_dep}'!G:G, '{sheet_dep}'!A:A, C$1, '{sheet_dep}'!E:E, A{row_poste}), 0)"
    postes_sheet[f'B{row_poste}'].number_format = '#,##0'
    
    # Colonne C : Somme des Patients (Somme de E de l'onglet effectifs conditionnée par l'année C1 et le poste A9+)
    postes_sheet[f'C{row_poste}'] = f"=IFERROR(SUMIFS('{sheet_eff}'!E:E, '{sheet_eff}'!A:A, C$1, '{sheet_eff}'!B:B, A{row_poste}), 0)"
    postes_sheet[f'C{row_poste}'].number_format = '#,##0'
    
    # Colonne D : Coût moyen par patient (Dépenses / Patients)
    postes_sheet[f'D{row_poste}'] = f"=IFERROR(B{row_poste} / C{row_poste}, 0)"
    postes_sheet[f'D{row_poste}'].number_format = '#,##0'
    
    row_poste += 1

# Recalculer le KPI global par rapport aux cellules injectées
postes_sheet['A6'] = f"=SUM(B9:B{row_poste-1})"

auto_adjust_columns(postes_sheet)

wb_dep.save(output_dep)
wb_dep.close()

## Onglet Pathologies

In [56]:
wb_dep = load_workbook(output_dep)

if 'Pathologies' not in wb_dep.sheetnames:
    pathos_sheet = wb_dep.create_sheet('Pathologies', 2)
else:
    pathos_sheet = wb_dep['Pathologies']
    pathos_sheet.delete_rows(1, pathos_sheet.max_row)

pathos_sheet.sheet_view.showGridLines = False

# FILTRES SUR PATHOLOGIES
add_filter(
    sheet=pathos_sheet, title='Année', value=int(annees[-1]),
    start_row=1, start_column=1, end_row=2, end_column=2,
    formula=f"=Filtres!$A$2:$A${len_dict['len_annee']}"
)

add_filter(
    sheet=pathos_sheet, title='Grand Poste', value=postes[0] if postes else 'N/A',
    start_row=1, start_column=5, end_row=2, end_column=6,
    formula=f"=Filtres!$C$2:$C${len_dict['len_dep_niv_1']}"
)

pathos_sheet['A3'] = "Analyse par pathologies"
pathos_sheet['A3'].font = Font(bold=True, size=14, color=COULEURS['cyan'])
pathos_sheet['A3'].fill = PatternFill(start_color=COULEURS['bleu_fonce'], fill_type='solid')
pathos_sheet.merge_cells('A3:E3')

pathos_sheet['A5'] = "Total dépenses"
pathos_sheet['A5'].font = Font(bold=True)

headers_patho = ['Pathologie', 'Dépenses', 'Patients', 'Coût/patient']
for col_idx, header in enumerate(headers_patho, 1):
    cell = pathos_sheet.cell(row=8, column=col_idx)
    cell.value = header
    cell.font = Font(bold=True, color='FFFFFF')
    cell.fill = PatternFill(start_color=COULEURS['bleu_clair'], fill_type='solid')
    cell.alignment = Alignment(horizontal='center')

row_patho = 9
for patho in pathos:
    pathos_sheet[f'A{row_patho}'] = patho
    pathos_sheet[f'A{row_patho}'].font = Font(size=10)
    
    pathos_sheet[f'B{row_patho}'] = f"=IFERROR(SUMIFS('{sheet_dep}'!G:G, '{sheet_dep}'!A:A, C$1, '{sheet_dep}'!E:E, F$1, '{sheet_dep}'!B:B, A{row_patho}), 0)"
    pathos_sheet[f'C{row_patho}'] = f"=IFERROR(SUMIFS('{sheet_dep}'!H:H, '{sheet_dep}'!A:A, C$1, '{sheet_dep}'!E:E, F$1, '{sheet_dep}'!B:B, A{row_patho}), 0)"
    pathos_sheet[f'D{row_patho}'] = f"=IFERROR(B{row_patho}/C{row_patho}, 0)"
    
    for col in ['B', 'C', 'D']:
        pathos_sheet[f'{col}{row_patho}'].number_format = '#,##0'
        pathos_sheet[f'{col}{row_patho}'].font = Font(size=10)
    row_patho += 1

pathos_sheet['A6'] = f"=IFERROR(SUM(B9:B{row_patho-1}), 0)"
pathos_sheet['A6'].font = Font(bold=True, size=14, color=COULEURS['bleu_fonce'])
pathos_sheet['A6'].number_format = '#,##0'

auto_adjust_columns(pathos_sheet)
pathos_sheet.column_dimensions['A'].width = 32
pathos_sheet.column_dimensions['B'].width = 22

chart2 = PieChart()
chart2.title = "Distribution par pathologie"
chart2.height = 12
chart2.width = 18

if row_patho > 9:
    chart2.add_data(Reference(pathos_sheet, min_col=2, min_row=9, max_row=row_patho-1), titles_from_data=False)
    chart2.set_categories(Reference(pathos_sheet, min_col=1, min_row=9, max_row=row_patho-1))
    if chart2.series:
        chart2.series[0].title = SeriesLabel(v="Dépenses")
    pathos_sheet.add_chart(chart2, "H5")

wb_dep.save(output_dep)
wb_dep.close()

## Onglet Effectifs

In [57]:
from openpyxl.styles import Font, PatternFill, Alignment

wb_eff = load_workbook(output_eff)

if 'Effectifs' in wb_eff.sheetnames:
    effectifs_sheet = wb_eff['Effectifs']
    
    # 0. SÉCURITÉ : Nettoyer proprement TOUS les anciens graphiques de l'onglet
    effectifs_sheet._charts.clear()
    
    # --- NETTOYAGE DES ANCIENNES TABLES SOLICITÉES ---
    # Nettoyage de l'ancienne table département colonnes J-K et de la matrice T:AK
    # On vide les cellules pour éviter les résidus visuels
    for row in range(5, 40):
        effectifs_sheet[f'J{row}'] = None
        effectifs_sheet[f'K{row}'] = None
        for col_idx in range(20, 38): # Colonnes T à AK
            effectifs_sheet.cell(row=row, column=col_idx, value=None)

    # 1. CRÉATION DU FILTRE DE PATHOLOGIE VISUEL (Zone B4)
    effectifs_sheet['A4'] = "Filtrer par Pathologie :"
    effectifs_sheet['A4'].font = Font(bold=True, color="000000")
    
    # Cellule B4 : contiendra le nom de la pathologie (ex: "Diabète" ou "Tous régimes")
    # Note : Tu pourras lier cette cellule B4 à ta liste déroulante de l'onglet Filtres via Excel !
    effectifs_sheet['B4'] = "Total consommants tous régimes" 
    effectifs_sheet['B4'].font = Font(bold=True, color="1F497D")
    effectifs_sheet['B4'].fill = PatternFill(start_color="E9EDF4", fill_type="solid")

    # 2. INJECTION + CALCUL DES DONNÉES FILTRÉES (Via l'état de B4)
    patho_filtre = effectifs_sheet['B4'].value
    
    # Application du filtre selon la sélection
    if patho_filtre and patho_filtre != "Total consommants tous régimes":
        df_filtered = df_effectifs[df_effectifs['patho_niv1'].str.strip() == patho_filtre.strip()]
    else:
        df_filtered = df_effectifs[df_effectifs['patho_niv1'].str.strip() != "Total consommants tous régimes"]

    # 3. RECONSTRUCTION DU TABLEAU 1 : Classes d'âges (Colonnes A et B)
    df_age = df_filtered.groupby('Classe d\'âge')['Effectif'].sum().reset_index()
    # Tri décroissant pour faire ressortir le Top des âges
    df_age = df_age.sort_values(by='Effectif', ascending=False)

    effectifs_sheet['A6'] = "Classe d'âge"
    effectifs_sheet['B6'] = "Effectif"
    for col in ['A', 'B']:
        effectifs_sheet[f'{col}6'].font = Font(bold=True, color='FFFFFF')
        effectifs_sheet[f'{col}6'].fill = PatternFill(start_color=COULEURS['bleu_clair'], fill_type='solid')

    row_age = 7
    for _, r in df_age.iterrows():
        effectifs_sheet[f'A{row_age}'] = r['Classe d\'âge']
        effectifs_sheet[f'B{row_age}'] = r['Effectif']
        effectifs_sheet[f'B{row_age}'].number_format = '#,##0'
        row_age += 1
        
    # Nettoyage des lignes du dessous si le nouveau tableau est plus court que l'ancien
    for r_clear in range(row_age, 30):
        effectifs_sheet[f'A{r_clear}'] = None
        effectifs_sheet[f'B{r_clear}'] = None

    # 4. RECONSTRUCTION DU TABLEAU 2 : Sexe (Colonnes D et E)
    df_sexe = df_filtered.groupby('Sexe')['Effectif'].sum().reset_index()
    
    effectifs_sheet['D6'] = "Sexe"
    effectifs_sheet['E6'] = "Effectif"
    for col in ['D', 'E']:
        effectifs_sheet[f'{col}6'].font = Font(bold=True, color='FFFFFF')
        effectifs_sheet[f'{col}6'].fill = PatternFill(start_color=COULEURS['bleu_clair'], fill_type='solid')

    row_sexe = 7
    for _, r in df_sexe.iterrows():
        effectifs_sheet[f'D{row_sexe}'] = r['Sexe']
        effectifs_sheet[f'E{row_sexe}'] = r['Effectif']
        effectifs_sheet[f'E{row_sexe}'].number_format = '#,##0'
        row_sexe += 1
        
    for r_clear in range(row_sexe, 15):
        effectifs_sheet[f'D{r_clear}'] = None
        effectifs_sheet[f'E{r_clear}'] = None

    # 5. --- NOUVEAU GRAPH : Top 10 Classes d'âge en fonction de la Pathologie (Placé en A22) ---
    chart_age = BarChart()
    chart_age.type = "col"
    chart_age.style = 10
    chart_age.title = f"Top 10 Classes d'âge - {patho_filtre if patho_filtre else 'Global'}"
    chart_age.y_axis.title = "Effectifs"
    chart_age.x_axis.title = "Classes d'âge"
    
    # On cible uniquement le Top 10 affiché dans les lignes 7 à 16 des colonnes A et B
    max_row_graph = min(row_age - 1, 16)
    
    data_age = Reference(effectifs_sheet, min_col=2, min_row=6, max_row=max_row_graph) # Colonne B (Effectifs)
    cats_age = Reference(effectifs_sheet, min_col=1, min_row=7, max_row=max_row_graph) # Colonne A (Ages)
    
    chart_age.add_data(data_age, titles_from_data=True)
    chart_age.set_categories(cats_age)
    chart_age.legend = None # Pas besoin de légende pour une seule série
    
    chart_age.height = 14
    chart_age.width = 20
    effectifs_sheet.add_chart(chart_age, "A22")

    # Ajustement automatique des tailles de colonnes restantes
    auto_adjust_columns(effectifs_sheet)

wb_eff.save(output_eff)
wb_eff.close()

Dépenses

In [76]:
if 'Depenses' in wb_dep.sheetnames:
    depenses_sheet = wb_dep['Depenses']
    
    for row in range(6, 150):
        for col in ['A', 'B', 'C', 'D', 'E', 'F']:
            depenses_sheet[f'{col}{row}'] = None

    condition_annee = f"(Donnees_Source!$C$2:$C${max_src}=$C$4)"
    condition_poste = f"IF($C$3=\"Tous\", 1, Donnees_Source!$B$2:$B${max_src}=$C$3)"
    
    formule_filter = f"FILTER(Donnees_Source!$A$2:$A${max_src}, {condition_annee} * {condition_poste}, \"Aucune donnée\")"
    depenses_sheet['A9'] = f"=SORT(UNIQUE({formule_filter}))"

    depenses_sheet['B9'] = (
        f"=IF(A9#=\"\", \"\", SUMIFS(Donnees_Source!$D$2:$D${max_src}, "
        f"Donnees_Source!$A$2:$A${max_src}, A9#, "
        f"Donnees_Source!$C$2:$C${max_src}, $C$4, "
        f"Donnees_Source!$B$2:$B${max_src}, IF($C$3=\"Tous\", \"*\", $C$3)))"
    )
    
    depenses_sheet['C9'] = (
        f"=IF(A9#=\"\", \"\", SUMIFS(Donnees_Source!$E$2:$E${max_src}, "
        f"Donnees_Source!$A$2:$A${max_src}, A9#, "
        f"Donnees_Source!$C$2:$C${max_src}, $C$4, "
        f"Donnees_Source!$B$2:$B${max_src}, IF($C$3=\"Tous\", \"*\", $C$3)))"
    )
    
    depenses_sheet['D9'] = f"=IF(C9#=0, 0, B9#/C9#)"

    for row in range(9, 100):
        depenses_sheet[f'B{row}'].number_format = '#,##0 €'
        depenses_sheet[f'C{row}'].number_format = '#,##0'
        depenses_sheet[f'D{row}'].number_format = '#,##0.00 €'

    depenses_sheet['A6'] = "Total dépenses :"
    depenses_sheet['A6'].font = Font(bold=True, size=11)
    depenses_sheet['B6'] = "=SUM(B9#)"
    depenses_sheet['B6'].font = Font(bold=True, size=12, color="1F497D")
    depenses_sheet['B6'].number_format = '#,##0 €'

    depenses_sheet['A8'] = "Pathologie"
    depenses_sheet['B8'] = "Dépenses"
    depenses_sheet['C8'] = "Patients"
    depenses_sheet['D8'] = "Coût/patient"

    for col in ['A', 'B', 'C', 'D']:
        cell = depenses_sheet[f'{col}8']
        cell.font = Font(bold=True, color='FFFFFF')
        cell.fill = PatternFill(start_color=COULEURS['bleu_clair'], fill_type='solid')

    for col in ['A', 'B', 'C', 'D']:
        depenses_sheet.column_dimensions[col].width = 30 if col != 'A' else 55

ws_source.sheet_state = 'hidden'

wb_dep.save(output_dep)
wb_dep.close()

NameError: name 'max_src' is not defined

Les graphiques :

In [ ]:
if 'Postes' in wb_dep.sheetnames:
    postes_sheet = wb_dep['Postes']
    
    # Détection dynamique de la fin du tableau
    row_poste = 9
    while postes_sheet[f'A{row_poste}'].value is not None:
        row_poste += 1
        
    chart_postes = BarChart()
    chart_postes.type = "bar"
    chart_postes.title = "Distribution par poste de dépenses"
    chart_postes.height = 12
    chart_postes.width = 18
    
    data_postes = Reference(postes_sheet, min_col=2, min_row=9, max_row=row_poste-1)
    cats_postes = Reference(postes_sheet, min_col=1, min_row=9, max_row=row_poste-1)
    
    chart_postes.add_data(data_postes, titles_from_data=False)
    chart_postes.set_categories(cats_postes)
    
    if chart_postes.series:
        chart_postes.series[0].title = SeriesLabel(v="Dépenses")
        
    postes_sheet.add_chart(chart_postes, "G5")
    auto_adjust_columns(postes_sheet)

# --- GRAPH 2 : Onglet Pathologies (Camembert / Pie) ---
if 'Pathologies' in wb_dep.sheetnames:
    pathos_sheet = wb_dep['Pathologies']
    
    # Détection dynamique de la fin du tableau
    row_patho = 9
    while pathos_sheet[f'A{row_patho}'].value is not None:
        row_patho += 1
        
    if row_patho > 9:
        chart_pathos = PieChart()
        chart_pathos.title = "Distribution des dépenses par pathologie"
        chart_pathos.height = 12
        chart_pathos.width = 18
        
        data_pathos = Reference(pathos_sheet, min_col=2, min_row=9, max_row=row_patho-1)
        cats_pathos = Reference(pathos_sheet, min_col=1, min_row=9, max_row=row_patho-1)
        
        chart_pathos.add_data(data_pathos, titles_from_data=False)
        chart_pathos.set_categories(cats_pathos)
        
        if chart_pathos.series:
            chart_pathos.series[0].title = SeriesLabel(v="Dépenses")
            
        pathos_sheet.add_chart(chart_pathos, "H5")
    
    auto_adjust_columns(pathos_sheet)

# Sauvegarde du classeur dépenses
wb_dep.save(output_dep)
wb_dep.close()
2. Bloc Graphiques des Effectifs (wb_eff)
Ce bloc gère l'onglet Effectifs avec l'histogramme des départements et le graphique empilé 100% Sexe/Patho.

Note : Pour que le graphique empilé fonctionne, on recalcule d'abord le tableau de données croisées df_pivot en lignes (à partir de la ligne 6 de l'onglet Effectifs).

Python
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill
from openpyxl.chart import BarChart, Reference

# === CHARGEMENT DU CLASSEUR EFFECTIFS ===
wb_eff = load_workbook(output_eff)

if 'Effectifs' in wb_eff.sheetnames:
    effectifs_sheet = wb_eff['Effectifs']
    
    # 1. Détection des lignes de la table des Départements (Table 4, Colonnes J et K)
    row_dept = 7
    while effectifs_sheet[f'J{row_dept}'].value is not None:
        row_dept += 1

    # 2. Injection + Calcul de la table Pivot Patho vs Sexe (Colonnes O, P, Q)
    df_pivot = df_effectifs.groupby(['patho_niv1', 'Sexe'])['Effectif'].sum().unstack().fillna(0)
    df_pivot['Total'] = df_pivot.sum(axis=1)
    df_pivot = df_pivot.sort_values(by='Total', ascending=False).head(10).drop(columns=['Total'])

    effectifs_sheet['O6'] = "Pathologie"
    effectifs_sheet['P6'] = "Homme"
    effectifs_sheet['Q6'] = "Femme"
    for col in ['O', 'P', 'Q']:
        effectifs_sheet[f'{col}6'].font = Font(bold=True, color='FFFFFF')
        effectifs_sheet[f'{col}6'].fill = PatternFill(start_color=COULEURS['bleu_clair'], fill_type='solid')

    row_pivot = 7
    for patho, rows in df_pivot.iterrows():
        effectifs_sheet[f'O{row_pivot}'] = patho
        effectifs_sheet[f'P{row_pivot}'] = rows.get('Homme', rows.get('H', rows.get('M', rows.iloc[0])))
        effectifs_sheet[f'Q{row_pivot}'] = rows.get('Femme', rows.get('F', rows.iloc[1] if len(rows) > 1 else 0))
        effectifs_sheet[f'P{row_pivot}'].number_format = '#,##0'
        effectifs_sheet[f'Q{row_pivot}'].number_format = '#,##0'
        row_pivot += 1

    # --- GRAPH 1 : Histogramme vertical (TOP 10 Département) ---
    chart_bar = BarChart()
    chart_bar.type = "col"
    chart_bar.style = 10
    chart_bar.title = "TOP 10 Effectifs par Département"
    chart_bar.y_axis.title = "Effectifs"
    chart_bar.x_axis.title = "Départements"

    data_dept = Reference(effectifs_sheet, min_col=11, min_row=6, max_row=row_dept-1)
    cats_dept = Reference(effectifs_sheet, min_col=10, min_row=7, max_row=row_dept-1)
    chart_bar.add_data(data_dept, titles_from_data=True)
    chart_bar.set_categories(cats_dept)
    chart_bar.legend = None
    chart_bar.height = 14
    chart_bar.width = 22
    effectifs_sheet.add_chart(chart_bar, "A22")

    # --- GRAPH 2 : Barres horizontales empilées 100% (Patho vs Sexe) ---
    chart_stack = BarChart()
    chart_stack.type = "bar"
    chart_stack.grouping = "percentStacked"
    chart_stack.overlap = 100
    chart_stack.title = "Répartition Top 10 Pathologies par Sexe (100%)"

    data_stack = Reference(effectifs_sheet, min_col=16, max_col=17, min_row=6, max_row=row_pivot-1)
    cats_stack = Reference(effectifs_sheet, min_col=15, min_row=7, max_row=row_pivot-1)
    chart_stack.add_data(data_stack, titles_from_data=True)
    chart_stack.set_categories(cats_stack)
    chart_stack.height = 14
    chart_stack.width = 22
    effectifs_sheet.add_chart(chart_stack, "J22")

    auto_adjust_columns(effectifs_sheet)

# Sauvegarde du classeur effectifs
wb_eff.save(output_eff)
wb_eff.close()

Effectifs :

In [68]:
from openpyxl.styles import Font, PatternFill
from openpyxl.chart import BarChart, Reference

wb_eff = load_workbook(output_eff)

if 'Effectifs' in wb_eff.sheetnames:
    effectifs_sheet = wb_eff['Effectifs']
    
    # 0. SÉCURITÉ : Nettoyer proprement TOUS les anciens graphiques pour éviter les superpositions
    effectifs_sheet._charts.clear()
    
    # --- 1. NETTOYAGE COMPLET DES COLONNES DEMANDÉES ---
    # Enlever la table Département / Effectif (Colonnes J et K)
    # Enlever la grande matrice Département / Pathologies (Colonnes T à AK)
    for row in range(5, 50):
        effectifs_sheet[f'J{row}'] = None
        effectifs_sheet[f'K{row}'] = None
        for col_idx in range(20, 38):  # Colonnes T à AK (20 à 37)
            effectifs_sheet.cell(row=row, column=col_idx, value=None)

    # --- 2. CONFIGURATION DE LA ZONE DE FILTRE PATHOLOGIE ---
    # On place le filtre en D4 / E4 pour ne pas gêner la table principale de gauche
    effectifs_sheet['D4'] = "Pathologie sélectionnée :"
    effectifs_sheet['D4'].font = Font(bold=True, color="000000")
    
    # Si la cellule est vide, on l'initialise avec la valeur globale par défaut
    if effectifs_sheet['E4'].value is None:
        effectifs_sheet['E4'] = "Total consommants tous régimes"
        
    effectifs_sheet['E4'].font = Font(bold=True, color="1F497D")
    effectifs_sheet['E4'].fill = PatternFill(start_color="E9EDF4", fill_type="solid")

    # Récupération de la valeur de la cellule de filtre
    patho_filtre = effectifs_sheet['E4'].value

    # --- 3. TRAITEMENT ET FILTRAGE DES DONNÉES (CLASSES D'ÂGE) ---
    # On applique le filtre choisi sur les données
    if patho_filtre and patho_filtre.strip() != "Total consommants tous régimes":
        df_age_filtered = df_effectifs[df_effectifs['patho_niv1'].str.strip() == patho_filtre.strip()]
    else:
        # Si aucun filtre ou "Total consommants", on garde tout sauf la ligne de agrégation globale
        df_age_filtered = df_effectifs[df_effectifs['patho_niv1'].str.strip() != "Total consommants tous régimes"]

    # On groupe par classe d'âge et on prend TOUTES les pathologies (sans limiter au Top 10)
    df_age = df_age_filtered.groupby('Classe d\'âge')['Effectif'].sum().reset_index()
    # Tri décroissant pour le confort visuel et le graphique
    df_age = df_age.sort_values(by='Effectif', ascending=False)

    # Écriture de la table Classe d'âge filtrée (Colonnes D et E)
    effectifs_sheet['D6'] = "Classe d'âge"
    effectifs_sheet['E6'] = "Effectif"
    for col in ['D', 'E']:
        effectifs_sheet[f'{col}6'].font = Font(bold=True, color='FFFFFF')
        effectifs_sheet[f'{col}6'].fill = PatternFill(start_color=COULEURS['bleu_clair'], fill_type='solid')

    row_age = 7
    for _, r in df_age.iterrows():
        effectifs_sheet[f'D{row_age}'] = r['Classe d\'âge']
        effectifs_sheet[f'E{row_age}'] = r['Effectif']
        effectifs_sheet[f'E{row_age}'].number_format = '#,##0'
        row_age += 1
        
    # Nettoyage des cellules du dessous pour effacer d'anciennes données plus longues
    for r_clear in range(row_age, 50):
        effectifs_sheet[f'D{r_clear}'] = None
        effectifs_sheet[f'E{r_clear}'] = None

    # --- 4. TABLEAU DES SEXES (CONSERVÉ EN GLOBAL SANS LE FILTRE PATHO) ---
    df_sexe_global = df_effectifs[df_effectifs['patho_niv1'].str.strip() != "Total consommants tous régimes"]
    df_sexe = df_sexe_global.groupby('Sexe')['Effectif'].sum().reset_index()
    
    effectifs_sheet['G6'] = "Sexe"
    effectifs_sheet['H6'] = "Effectif"
    for col in ['G', 'H']:
        effectifs_sheet[f'{col}6'].font = Font(bold=True, color='FFFFFF')
        effectifs_sheet[f'{col}6'].fill = PatternFill(start_color=COULEURS['bleu_clair'], fill_type='solid')

    row_sexe = 7
    for _, r in df_sexe.iterrows():
        effectifs_sheet[f'G{row_sexe}'] = r['Sexe']
        effectifs_sheet[f'H{row_sexe}'] = r['Effectif']
        effectifs_sheet[f'H{row_sexe}'].number_format = '#,##0'
        row_sexe += 1
        
    for r_clear in range(row_sexe, 20):
        effectifs_sheet[f'G{r_clear}'] = None
        effectifs_sheet[f'H{r_clear}'] = None

    # --- 5. CRÉATION DU GRAPHIQUE TOP 10 CLASSE D'ÂGE FONCTION DE LA PATHOLOGIE ---
    chart_age = BarChart()
    chart_age.type = "col"
    chart_age.style = 10
    chart_age.title = f"Top 10 Classes d'âge - {patho_filtre}"
    chart_age.y_axis.title = "Effectifs"
    chart_age.x_axis.title = "Classes d'âge"
    
    # On limite le graphique au Top 10 (lignes 7 à 16 de notre tableau trié)
    max_row_graph = min(row_age - 1, 16)
    
    # Références sur les colonnes D (Classes d'âge) et E (Effectifs filtrés)
    data_age = Reference(effectifs_sheet, min_col=5, min_row=6, max_row=max_row_graph)  # Colonne E
    cats_age = Reference(effectifs_sheet, min_col=4, min_row=7, max_row=max_row_graph)  # Colonne D
    
    chart_age.add_data(data_age, titles_from_data=True)
    chart_age.set_categories(cats_age)
    chart_age.legend = None  # Pas de légende nécessaire (une seule série de données)
    
    chart_age.height = 14
    chart_age.width = 22
    
    # Positionnement du nouveau graphique en dessous des tableaux
    effectifs_sheet.add_chart(chart_age, "D22")

    # Ajustement des largeurs de colonnes
    auto_adjust_columns(effectifs_sheet)

wb_eff.save(output_eff)
wb_eff.close()

In [ ]:
from openpyxl import load_workbook, Workbook

In [ ]:
output_final = "Rapport_Consolide_SNDS_2023.xlsx"

def fusionner_classeurs_memoire(wb_dep, wb_eff, file_final):
    """
    Fusionne les onglets directement depuis les objets Workbook ouverts en mémoire
    pour éviter les erreurs de fichiers non synchronisés.
    """
    print("🔄 Début de la fusion des classeurs depuis la mémoire...")
    
    # 1. Initialiser le classeur final
    wb_final = Workbook()
    if "Sheet" in wb_final.sheetnames:
        del wb_final["Sheet"]
    
    # 2. Utiliser directement les structures d'onglets passées en arguments
    structure_onglets = [
        {'nom': 'Filtres', 'source': wb_dep},
        {'nom': 'Postes', 'source': wb_dep},
        {'nom': 'Pathologies', 'source': wb_dep},
        {'nom': 'Effectifs', 'source': wb_eff},
        {'nom': 'Depenses', 'source': wb_dep}
    ]
    
    for item in structure_onglets:
        nom = item['nom']
        wb_source = item['source']
        
        # Vérification de sécurité dans l'objet en mémoire
        if nom in wb_source.sheetnames:
            print(f"📁 Copie de l'onglet : {nom}")
            ws_source = wb_source[nom]
            ws_cible = wb_final.create_sheet(title=nom)
            
            # --- Copie des valeurs et styles ---
            for row in ws_source.iter_rows():
                for cell in row:
                    new_cell = ws_cible.cell(row=cell.row, column=cell.column, value=cell.value)
                    if cell.has_style:
                        new_cell.font = cell.font
                        new_cell.fill = cell.fill
                        new_cell.number_format = cell.number_format
                        new_cell.alignment = cell.alignment
                        new_cell.border = cell.border
            
            # --- Copie des fusions ---
            for merged_range in list(ws_source.merged_cells.ranges):
                ws_cible.merge_cells(str(merged_range))
                
            # --- Re-création des listes déroulantes ---
            for dv_src in ws_source.data_validations.dataValidation:
                dv_new = DataValidation(type=dv_src.type, formula1=dv_src.formula1, allow_blank=dv_src.allow_blank)
                ws_cible.add_data_validation(dv_new)
                for sqref in dv_src.sqref.ranges:
                    dv_new.add(str(sqref))
                        
            # --- Copie des graphiques ---
            for chart in ws_source._charts:
                ws_cible.add_chart(chart)
                
            # --- Largeurs de colonnes ---
            for col_name, col_dim in ws_source.column_dimensions.items():
                ws_cible.column_dimensions[col_name].width = col_dim.width
            
            ws_cible.sheet_view.showGridLines = ws_source.sheet_view.showGridLines
        else:
            print(f"⚠️ Onglet '{nom}' introuvable dans son classeur mémoire.")

    # 3. Sauvegarde unique du fichier consolidé (Boucle infinie supprimée)
    wb_final.save(file_final)
    wb_final.close()
    print(f"✅ Fusion terminée avec succès ! Fichier disponible : {file_final}")

## Sauvegarde finale

In [69]:
# 1. Sauvegarde des classeurs individuels (chacun sa variable)
wb_dep.save(output_dep)
wb_eff.save(output_eff)

# 2. Fermeture des classeurs (les parenthèses doivent rester totalement vides)
wb_dep.close()
wb_eff.close()